# Cursul 7. Variabile derivate și model
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

Analiza exploratorie (C6) a găsit trei tipare: SCI scade cu distanța, este mai mare peste o graniță comună și crește cu migrația. Tiparele se suprapun: țările vecine sunt și apropiate, și au mai mulți migranți între ele. Un **model** le include simultan și estimează ce rămâne din fiecare când celelalte sunt ținute constante.

Înainte de model, variabilele se pregătesc (**ingineria variabilelor**, engl. *feature engineering*):

| Etapa | Ce face | Pe datele SCI |
|---|---|---|
| 1. Construcția | variabile noi din cele existente | rata migranților, distanța dintre țări, granița comună |
| 2. Transformarea | forma distribuției | logaritmul |
| 3. Scalarea | aceeași scară pentru toate | scorul z |
| 4. Extragerea | mai mulți indicatori într-unul | gruparea regiunilor (k-means) |
| 5. Selecția | setul final, fără redundanță | corelațiile predictorilor, VIF |

Apoi modelul: trei modele, construite pe rând. M1 și M2 au rezultatul complet și reziduurile (secțiunile 6-8). O analiză de rețea mică (secțiunea 9) arată ce lipsește din M2, iar secțiunea 10 îl corectează: efectele de țară (M3). Un model pe date observaționale descrie **asocieri**, nu cauze.

Fiecare pas are o celulă, iar unde rezultatul se vede pe hartă, o hartă.

## 0. Pregătirea
### 0.1. Bibliotecile

In [ ]:
from urllib.request import urlretrieve
from sqlalchemy import create_engine
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
import statsmodels.formula.api as smf
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")

### 0.2. Datele, din baza de date a cursului
Datele stau în baza de date `dsw.db`, construită în C5. Caietul o descarcă din depozitul cursului (GitHub), deci rulează de oriunde, cu o conexiune la internet. Apoi tabelele se citesc cu SQL, ca în C5: regiunile, perechile de regiuni și perechile de țări.

In [ ]:
DATE = "https://raw.githubusercontent.com/dsw-ubb-2026/-DSW2026/main/curs/data/"   # depozitul cursului

urlretrieve(DATE + "dsw.db", "dsw.db")                  # baza de date din C5, salvată lângă caiet
engine = create_engine("sqlite:///dsw.db")              # conexiunea la baza de date

regiuni = pd.read_sql("select * from regiuni", engine)            # un rând pe regiune
perechi = pd.read_sql("select * from perechi_regiuni", engine)    # un rând pe pereche de regiuni
tari = pd.read_sql("select * from perechi_tari", engine)          # un rând pe pereche de țări

len(regiuni), len(perechi), len(tari)

### 0.3. Harta regiunilor
Contururile regiunilor (GISCO), ca în C3 și C6. Fișierul se descarcă lângă caiet, apoi geopandas îl citește.

In [ ]:
urlretrieve("https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_20M_2024_3035_LEVL_2.geojson", "regiuni.geojson")
harta = gpd.read_file("regiuni.geojson")                # un rând pe regiune, cu conturul ei
harta = harta.rename(columns={"NUTS_ID": "regiune"})[["regiune", "geometry"]]
harta = harta.merge(regiuni, on="regiune")              # contururile, cu datele regiunilor

In [ ]:
ax = harta.plot(color="#dddddd", edgecolor="white", figsize=(9, 7))
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Regiunile NUTS 2")
ax.set_axis_off()

### 0.4. Distanța dintre regiuni
Svalbard se exclude (decizia din C6: numitor foarte mic). Distanța dintre două regiuni este distanța dintre centrele lor, ca în C6.

In [ ]:
perechi = perechi[(perechi["regiune_1"] != "NO0B") & (perechi["regiune_2"] != "NO0B")]   # fără Svalbard

# centrul fiecărei regiuni; harta are coordonatele în metri
centre = harta.set_index("regiune").centroid

# distanța dintre centrele celor două regiuni din fiecare pereche, în kilometri (ca în C6)
centru_1 = centre[perechi["regiune_1"]].values
centru_2 = centre[perechi["regiune_2"]].values
perechi["distanta_km"] = centru_1.distance(centru_2) / 1000
perechi[["regiune_1", "regiune_2", "distanta_km", "sci"]].head()

## 1. Construcția: variabile noi din cele existente
Modelul se face pe **perechi de țări**, pentru că migrația există doar între țări. Tabelul `tari` are SCI-ul și migranții. Îi lipsesc trei variabile, construite pe rând: rata migranților, distanța dintre țări și granița comună.

### 1.1. Rata migranților
Numărul de migranți depinde de mărimea țărilor (C6). Împărțit la populația celor două țări, devine o **rată**, comparabilă între țări mari și mici. Întâi, populația fiecărei țări:

In [ ]:
bm = pd.read_csv(DATE + "raw/bm.csv", keep_default_na=False, na_values=[""])   # NA = Namibia, nu valoare lipsă
pop = bm[(bm["indicator"] == "populatie") & (bm["an"] == 2024)].set_index("tara")["valoare"]
pop[["RO", "DE", "LU"]]

Apoi rata: migranții la 1.000 de locuitori ai celor două țări. Se păstrează doar perechile cu migrația cunoscută.

In [ ]:
tari = tari.dropna(subset=["migranti"]).copy()           # doar perechile cu migrația cunoscută

# migranți la 1.000 de locuitori ai celor două țări (map: populația fiecărei țări din pereche)
tari["la_mie"] = 1000 * tari["migranti"] / (tari["tara_1"].map(pop) + tari["tara_2"].map(pop))
tari[["tara_1", "tara_2", "migranti", "la_mie"]].sort_values("la_mie", ascending=False).head()

### 1.2. Distanța dintre două țări
Distanța dintre două țări se construiește din distanțele dintre regiunile lor: **mediana** distanțelor dintre toate perechile de regiuni ale celor două țări. Mediana nu e trasă de regiunile îndepărtate (insulele, teritoriile de peste mări).

Întâi, o **cheie** a perechii de țări: cele două coduri, în ordine alfabetică, legate cu o liniuță („AT-RO”). În `tari`, codurile sunt deja în ordine alfabetică. În `perechi`, aceeași pereche poate apărea și ca „RO, AT”, deci codurile se ordonează întâi (`sorted`).

In [ ]:
tari["cheie"] = tari["tara_1"] + "-" + tari["tara_2"]          # "AT" și "RO" -> "AT-RO"

# în perechi: întâi cele două coduri în ordine alfabetică, apoi legate cu "-"
perechi["cheie"] = ["-".join(sorted([a, b])) for a, b in zip(perechi["tara_1"], perechi["tara_2"])]
perechi[["tara_1", "tara_2", "cheie"]].head(3)

Un exemplu: Austria are 9 regiuni, România 8, deci între ele sunt 72 de perechi de regiuni. Distanța dintre țări este mediana celor 72 de distanțe.

In [ ]:
perechi[perechi["cheie"] == "AT-RO"]["distanta_km"].describe().round(0)   # cele 72 de distanțe: count, medie, mediană (50%)

Același calcul pentru toate perechile de țări, cu `groupby`:

In [ ]:
distanta_tari = perechi.groupby("cheie")["distanta_km"].median()     # o distanță pe pereche de țări
tari["distanta_km"] = tari["cheie"].map(distanta_tari)
tari[["tara_1", "tara_2", "distanta_km"]].head()

### 1.3. Granița comună
Două țări sunt vecine dacă se ating. Pașii, fiecare cu harta lui:
1. regiunile se lipesc în țări (`dissolve`);
2. fiecare țară se lărgește cu 10 km (`buffer`), ca să acopere micile goluri dintre contururile simplificate;
3. se caută țările care se ating (`sjoin`).

**Pasul 1.** `dissolve("tara")` unește toate regiunile cu același cod de țară într-o singură formă.

In [ ]:
contur = harta[["tara", "geometry"]].dissolve("tara", as_index=False)    # o formă pe țară
ax = contur.plot(cmap="tab20", edgecolor="white", figsize=(9, 7))
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Țările, lipite din regiuni (dissolve)")
ax.set_axis_off()

**Pasul 2.** `buffer(10_000)` lărgește fiecare țară cu 10.000 de metri (unitatea hărții este metrul). Pe România:

In [ ]:
ro = contur[contur["tara"] == "RO"]                    # România
ax = ro.buffer(10_000).plot(color="#f4a261", figsize=(7, 5))     # lărgită cu 10 km (portocaliu)
ro.plot(ax=ax, color="#2a9d8f")                        # conturul inițial (verde)
ax.set_title("România și România lărgită cu 10 km")
ax.set_axis_off()

**Pasul 3.** `sjoin` caută, pentru fiecare țară lărgită (stânga), țările pe care le atinge (dreapta). Fiecare rând al rezultatului este o atingere: `tara_left` atinge `tara_right`. Fiecare țară se atinge și cu ea însăși.

In [ ]:
largit = contur.copy()
largit["geometry"] = contur.buffer(10_000)                         # toate țările, lărgite cu 10 km
atingeri = gpd.sjoin(largit, contur, predicate="intersects")       # cine pe cine atinge

vecini_ro = atingeri[atingeri["tara_left"] == "RO"]["tara_right"]  # pe cine atinge România
list(vecini_ro)

In [ ]:
ax = contur.plot(color="#eeeeee", edgecolor="white", figsize=(9, 7))
contur[contur["tara"].isin(vecini_ro)].plot(ax=ax, color="#f4a261")   # vecinii (portocaliu)
contur[contur["tara"] == "RO"].plot(ax=ax, color="#2a9d8f")           # România (verde)
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="România și vecinii ei")
ax.set_axis_off()

**Variabila.** Granița este o variabilă **binară**: 1 pentru țările vecine, 0 în rest. Atingerile devin un tabel de perechi cu `granita = 1`, lipit de `tari` cu `merge` (C4); perechile care nu se ating rămân fără valoare și primesc 0.

In [ ]:
vecini = atingeri.rename(columns={"tara_left": "tara_1", "tara_right": "tara_2"})[["tara_1", "tara_2"]]
vecini["granita"] = 1                                              # perechile care se ating

tari = tari.merge(vecini, on=["tara_1", "tara_2"], how="left")     # granita = 1 sau lipsă
tari["granita"] = tari["granita"].fillna(0).astype(int)            # lipsă -> 0: nu se ating
tari[tari["granita"] == 1][["tara_1", "tara_2"]].head(10)

**Exercițiu.** Câte perechi de țări vecine are tabelul?

In [ ]:
# Suma coloanei granita (0 sau 1): câte perechi au valoarea 1

## 2. Transformarea: logaritmul
SCI, distanța și rata migranților au coadă lungă spre dreapta (C6). Logaritmul le aduce aproape de simetrie și transformă relațiile multiplicative în relații liniare. Granița (0 sau 1) nu se logaritmează: nu are coadă, iar log 0 nu există.

Asimetria înainte și după logaritm:

In [ ]:
# asimetria fiecărei variabile, înainte -> după logaritm (0 = simetric)
for coloana in ["sci", "distanta_km", "la_mie"]:
    print(coloana, "asimetria:", round(tari[coloana].skew(), 2), "->", round(np.log10(tari[coloana]).skew(), 2))

Pe SCI, diferența se vede în histogramă:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
sns.histplot(tari["sci"], ax=ax[0]).set(title="SCI")                  # coadă lungă
sns.histplot(np.log10(tari["sci"]), ax=ax[1]).set(title="log10 SCI")  # aproape simetric

Coloanele noi, pe logaritm:

In [ ]:
# logaritmul în baza 10, ca în C6
tari["log_sci"] = np.log10(tari["sci"])
tari["log_distanta"] = np.log10(tari["distanta_km"])
tari["log_migranti"] = np.log10(tari["migranti"])
tari["log_la_mie"] = np.log10(tari["la_mie"])

In [ ]:
sns.regplot(tari, x="log_distanta", y="log_sci");          # pe log-log, relația e aproape o dreaptă

**Exercițiu.** Același grafic pe valorile brute, fără logaritm. Se mai vede o dreaptă?

In [ ]:
# sns.regplot pe coloanele distanta_km și sci, fără logaritm

**Același eșantion pentru toate modelele.** Modelele se compară doar pe exact aceleași perechi. Perechile cu o valoare lipsă se scot o singură dată, aici; tabelul `date` este cel folosit de aici înainte.

In [ ]:
date = tari.dropna(subset=["log_sci", "log_distanta", "log_la_mie", "granita"]).copy()   # perechile complete
len(date)                                                  # câte perechi intră în toate modelele

## 3. Scalarea: scorul z
Distanța (log) merge de la aproximativ 2 la 3,6, rata migranților (log) de la −3 la 1: scări diferite. **Scorul z** aduce fiecare variabilă la media 0 și abaterea standard 1: z = (x − media) / abaterea standard. Forma distribuției nu se schimbă, doar unitatea. Cu toate variabilele în scor z, coeficienții modelului se pot compara între ei (secțiunea 6.3).

In [ ]:
def scor_z(x):
    """Scorul z: câte abateri standard este fiecare valoare față de medie."""
    return (x - x.mean()) / x.std()

In [ ]:
for coloana in ["log_sci", "log_distanta", "log_la_mie"]:
    date["z_" + coloana] = scor_z(date[coloana])           # coloana nouă: z_log_sci, z_log_distanta, ...

date[["log_distanta", "z_log_distanta", "log_la_mie", "z_log_la_mie"]].describe().round(2)

**Exercițiu.** Asimetria ratei migranților înainte și după scorul z. Scalarea schimbă forma?

In [ ]:
# skew() pe log_la_mie și pe z_log_la_mie, separate prin virgulă

## 4. Extragerea: gruparea regiunilor (clustering)
**Extragerea** rezumă mai mulți indicatori într-o singură variabilă. Analiza componentelor principale (PCA) dă un scor numeric; **gruparea** (engl. *cluster analysis*) dă o categorie: regiunile asemănătoare pe toți indicatorii ajung în aceeași grupă. Rezultatul este o variabilă derivată nouă, „tipul de regiune”, utilă unui proiect pe regiuni. Gruparea se face pe **regiuni**, nu pe perechi.

**k-means** împarte regiunile în k grupe: fiecare regiune merge la cel mai apropiat centru, apoi fiecare centru se mută în media regiunilor lui, până nu se mai schimbă nimic.

### 4.1. Indicatorii, pe aceeași scară
Cinci indicatori ai regiunilor. Densitatea are coadă lungă, deci intră pe logaritm (secțiunea 2).

In [ ]:
indicatori = ["pib_locuitor", "studii_superioare", "somaj", "varsta_medie", "log_densitate"]

reg = regiuni.copy()
reg["log_densitate"] = np.log10(reg["densitate"])
reg = reg.dropna(subset=indicatori)                         # doar regiunile cu toți indicatorii
len(reg)

Distanța dintre regiuni se calculează pe toți indicatorii deodată, deci indicatorii trebuie să aibă aceeași scară: **scorul z este obligatoriu**. Altfel, PIB-ul (zeci de mii de euro) ar decide singur grupele.

In [ ]:
z = scor_z(reg[indicatori])                                 # scorul z din secțiunea 3, pe fiecare coloană
z.describe().round(2)                                       # media 0, abaterea standard 1, la toți

### 4.2. Câte grupe?
**Inerția** este suma distanțelor (la pătrat) de la fiecare regiune la centrul grupei ei. Scade mereu când k crește; se caută punctul după care scade mult mai încet (**cotul**).

In [ ]:
# inerția pentru k = 1, 2, ..., 8 grupe
inertie = [KMeans(n_clusters=k, n_init=10, random_state=1).fit(z).inertia_ for k in range(1, 9)]
plt.plot(range(1, 9), inertie, marker="o")
plt.xlabel("numărul de grupe, k"); plt.ylabel("inerția");

### 4.3. Patru grupe
`random_state=1` face rezultatul același la fiecare rulare; `n_init=10` repornește algoritmul de 10 ori și păstrează varianta cea mai bună.

In [ ]:
grupare = KMeans(n_clusters=4, n_init=10, random_state=1).fit(z)
reg["grupa"] = grupare.labels_
reg["grupa"].value_counts()                                 # câte regiuni are fiecare grupă

Profilul fiecărei grupe: mediana indicatorilor, pe scara inițială. După profil se dă un nume fiecărei grupe.

In [ ]:
# mediana fiecărui indicator, pe grupă (densitatea pe scara inițială, nu pe logaritm)
reg.groupby("grupa")[["pib_locuitor", "studii_superioare", "somaj", "varsta_medie", "densitate"]].median().round(1)

In [ ]:
harta["grupa"] = harta["regiune"].map(reg.set_index("regiune")["grupa"])   # grupa fiecărei regiuni, pe hartă
ax = harta.plot(column="grupa", categorical=True, cmap="Set2", legend=True, figsize=(9, 7),
                missing_kwds={"color": "#eeeeee"})
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Patru tipuri de regiuni (k-means pe cinci indicatori)")
ax.set_axis_off()

Inerția nu are un cot foarte clar: k = 4 este o alegere, justificată prin grupe care se pot descrie. Numerele grupelor sunt doar etichete și pot ieși în altă ordine; grupele se numesc după profil. Din 298 de regiuni intră 266: celelalte nu au toți indicatorii (Elveția, Norvegia, Balcanii de Vest).

| Tipul de regiune | Regiuni | PIB/loc. | Studii superioare (%) | Șomaj (%) | Vârsta medie | Densitate | Exemple |
|---|---|---|---|---|---|---|---|
| metropolitane | 36 | 60.100 | 50,0 | 4,7 | 41,4 | 740 | capitalele: Praga, Bratislava, Budapesta, Varșovia, București-Ilfov, Madrid, Stockholm |
| dezvoltate | 119 | 38.300 | 34,6 | 3,7 | 45,1 | 139 | Germania, Austria, nordul Italiei, Polonia vestică |
| periferice, îmbătrânite | 90 | 27.350 | 27,9 | 7,7 | 45,8 | 65 | Grecia, sudul Spaniei și al Italiei, Franța rurală, România fără București, Bulgaria |
| tinere, cu șomaj ridicat | 21 | 20.500 | 22,8 | 10,3 | 34,2 | 140 | Turcia, teritoriile franceze de peste mări, Ceuta |

România se împarte: București-Ilfov este în grupa metropolelor, celelalte șapte regiuni în grupa periferiilor. Gruparea nu știe nimic despre țări sau hartă, dar capitalele ies împreună. Ca la comunitățile din C6, grupele arată *că* regiunile se aseamănă, nu *de ce*.

In [ ]:
reg[reg["tara"] == "RO"][["denumire", "grupa"]]             # regiunile României și grupele lor

**De încercat.** Aceeași grupare **fără** scorul z: `KMeans(n_clusters=4, n_init=10, random_state=1).fit(reg[indicatori])`, apoi profilul grupelor. Ce indicator decide acum grupele? Apoi `n_clusters=3` și `n_clusters=6`.

## 5. Selecția: redundanța predictorilor
Înapoi la perechile de țări. Doi predictori care spun aproape același lucru nu se pot separa în model: coeficienții lor devin instabili (**multicoliniaritate**). Prima verificare: corelațiile dintre predictori.

In [ ]:
predictori = ["log_distanta", "log_la_mie", "granita"]     # variabilele explicative ale modelului
date[predictori].corr().round(2)                           # corelația fiecărei perechi de predictori

A doua: **VIF** (factorul de inflație a varianței). Fiecare predictor se explică prin ceilalți; dacă R² e mare, predictorul e redundant. VIF = 1 / (1 − R²). Sub 5: acceptabil; între 5 și 10: atenție; peste 10: unul dintre predictori se scoate.

In [ ]:
for v in predictori:
    ceilalti = " + ".join(x for x in predictori if x != v)
    r2 = smf.ols(f"{v} ~ {ceilalti}", data=date).fit().rsquared      # cât de bine îl explică ceilalți
    print(v, "VIF =", round(1 / (1 - r2), 2))

Toate valorile sunt sub 2. Distanța și granița sunt legate (r = −0,63: țările vecine sunt apropiate), dar nu atât încât să nu se poată separa. Toți trei predictorii rămân.

## 6. Modelul
### 6.1. O dreaptă: tipar și reziduu
Un **model liniar** scrie fiecare valoare ca tipar plus reziduu: log SCI = nivel + pantă × log distanță + reziduu. **Metoda celor mai mici pătrate** alege nivelul și panta care fac suma pătratelor reziduurilor cât mai mică. **R²** este partea din variația lui log SCI explicată de model, între 0 și 1.

În formula lui `smf.ols`, la stânga lui `~` stă variabila explicată, la dreapta predictorii.

In [ ]:
m1 = smf.ols("log_sci ~ log_distanta", data=date).fit()
m1.params                                                  # nivelul (Intercept) și panta

In [ ]:
m1.rsquared                                                # cât explică distanța singură

### 6.2. Trei variabile, fiecare cu o explicație
Un predictor intră în model doar dacă există un motiv teoretic pentru el și o direcție așteptată:

| Variabila | De ce ar conta | Semnul așteptat |
|---|---|---|
| distanța | contactul costă mai mult la distanță mare (modelul gravitațional) | negativ |
| rata migranților | cei plecați păstrează legăturile cu cei de acasă (rețelele de migrație) | pozitiv |
| granița comună | contact zilnic, istorie și minorități comune | pozitiv |

Secțiunea 5 a arătat că cele trei nu se suprapun (VIF sub 2). **M2** le conține pe toate trei și se compară cu M1, distanța singură.

In [ ]:
m2 = smf.ols("log_sci ~ log_distanta + log_la_mie + granita", data=date).fit()

modele = pd.DataFrame({"M1": m1.params, "M2": m2.params})   # câte o coloană pe model
modele.loc["R²"] = [m1.rsquared, m2.rsquared]
modele.round(2)

- Distanța are semn negativ în ambele modele, dar coeficientul ei scade de la −0,76 (M1) la −0,30 (M2): o parte din „efectul distanței” din M1 era migrație, pentru că țările apropiate au mai mulți migranți între ele.
- R² crește de la 0,16 la 0,53: migrația și granița explică mult peste distanță.
- Toate trei au semnul așteptat.

**Exercițiu.** Același model cu **numărul** de migranți (`log_migranti`) în locul ratei. Cât este R²? Ce arată despre construcția variabilelor din secțiunea 1?

In [ ]:
# Formula lui M2, cu log_migranti în locul lui log_la_mie; apoi rsquared pentru ambele modele

Cu numărul de migranți, R² este 0,26; cu rata, 0,53. Aceeași informație, construită altfel, explică de două ori mai mult: acesta este rostul variabilelor derivate.

### 6.3. Care variabilă contează mai mult?
Coeficienții din M2 au unități diferite. Pe scorurile z (secțiunea 3), fiecare coeficient spune cu câte abateri standard se schimbă log SCI când predictorul crește cu o abatere standard, deci se pot compara.

In [ ]:
m2z = smf.ols("z_log_sci ~ z_log_distanta + z_log_la_mie + granita", data=date).fit()   # M2, pe scorurile z
m2z.params.round(2)

Rata migranților are coeficientul cel mai mare (0,62), de aproape patru ori mai mare decât distanța (−0,16). Între perechile de țări, migrația contează mai mult decât distanța, odată ce amândouă sunt în model. Granița rămâne pe scara 0/1: coeficientul ei (0,22) este diferența dintre țările vecine și celelalte, în abateri standard ale lui log SCI.

## 7. Rezultatul complet și interpretarea
### 7.1. Ce conține `summary()`
`m2.summary()` dă tot rezultatul modelului, în trei blocuri: modelul în ansamblu (sus), coeficienții (mijloc), diagnosticul reziduurilor (jos).

In [ ]:
m2.summary()

**Sus: modelul în ansamblu.**
- `No. Observations: 263`, `Df Model: 3` (trei predictori), `Df Residuals: 259` (263 − 3 − 1, pentru constantă).
- `R-squared: 0.530`: modelul explică 53% din variația lui log SCI. `Adj. R-squared: 0.525`: același R², penalizat pentru numărul de predictori. R² crește la orice predictor adăugat, chiar și unul fără legătură; R² ajustat crește doar dacă predictorul aduce mai mult decât ar aduce întâmplarea.
- `F-statistic: 97.52`, `Prob (F-statistic): 2.88e-42`: testul ipotezei că niciun predictor nu contează. Valoarea p e practic 0: modelul în ansamblu explică ceva.
- `AIC: 192.2`, `BIC: 206.5`: pentru compararea modelelor pe **aceleași** date; valoarea mai mică e mai bună.

**Mijloc: coeficienții.** Pe fiecare rând: `coef` (estimarea), `std err` (eroarea standard: cât de precisă este estimarea), `t` (coef împărțit la eroarea standard), `P>|t|` (valoarea p, ca în C6), `[0.025 0.975]` (intervalul de încredere de 95%).

| | coef | std err | t | P>\|t\| | interval 95% | Citirea |
|---|---|---|---|---|---|---|
| Intercept | 5,42 | 0,33 | 16,6 | 0,000 | 4,77 … 6,06 | log SCI prezis la 1 km și o rată de 1 la mie: o extrapolare, fără sens direct |
| log_distanta | −0,30 | 0,10 | −2,9 | 0,004 | −0,51 … −0,10 | semnificativ: SCI scade cu distanța, la aceeași migrație și graniță |
| log_la_mie | 0,44 | 0,03 | 13,7 | 0,000 | 0,38 … 0,51 | cel mai puternic și cel mai precis: t de 13,7 |
| granita | 0,11 | 0,09 | 1,3 | 0,210 | −0,06 … 0,28 | nesemnificativ: p peste 0,05, intervalul conține 0 |

Regula practică: un t peste 2 în valoare absolută corespunde unei valori p sub 0,05.

**Jos: diagnosticul reziduurilor.** Modelul liniar presupune reziduuri aproximativ normale, independente, cu aceeași împrăștiere.
- `Omnibus` și `Jarque-Bera` (ambele cu p < 0,001): reziduurile nu sunt perfect normale. `Skew: 0.65`: câteva perechi mult peste model (Islanda). `Kurtosis: 3.57`: puțin peste 3, valoarea distribuției normale.
- `Durbin-Watson: 2.02`: aproape de 2, fără autocorelație între rândurile vecine; contează mai ales la serii de timp.
- `Cond. No.: 54`: mic; valorile mari (statsmodels avertizează peste 1.000) ar semnala multicoliniaritate.
- Erorile standard presupun observații independente. Perechile nu sunt independente (Germania apare în 23 de perechi), deci erorile standard sunt subestimate; corecția uzuală este gruparea pe țară.

### 7.2. Coeficienții, în cuvinte
Când ambele părți sunt pe logaritm, coeficientul este o **elasticitate**: dacă predictorul se înmulțește cu 2, SCI se înmulțește cu 2 la puterea coeficientului. Pentru granița (0 sau 1), SCI se înmulțește cu 10 la puterea coeficientului.

In [ ]:
b = m2.params
print("distanța de două ori mai mare:   SCI ×", round(2 ** b["log_distanta"], 2))
print("rata migranților de două ori:    SCI ×", round(2 ** b["log_la_mie"], 2))
print("graniță comună:                  SCI ×", round(10 ** b["granita"], 2))

O distanță de două ori mai mare merge cu un SCI cu aproximativ 19% mai mic; o rată a migranților de două ori mai mare, cu un SCI cu aproximativ 36% mai mare; țările vecine au un SCI cu aproximativ 29% mai mare, la aceeași distanță și migrație, dar intervalul graniței conține 0.

### 7.3. Interpretarea, ca într-un raport
Pe 263 de perechi de țări europene, un model liniar cu distanța, rata migranților și granița comună explică 53% din variația lui log SCI (R² ajustat 0,52). **Migrația este predictorul cel mai puternic**: la aceeași distanță și graniță, o rată a migranților de două ori mai mare merge cu un SCI cu aproximativ 36% mai mare (coeficient 0,44, interval de 95% între 0,38 și 0,51). **Distanța rămâne semnificativă, dar mult mai slabă decât părea singură**: coeficientul ei scade de la −0,76 în M1 la −0,30 în M2, pentru că o parte din efectul ei trecea prin migrație. **Granița comună nu adaugă un efect clar** odată ce distanța și migrația sunt în model (p = 0,21). Rezultatele sunt asocieri între perechi de țări, nu efecte cauzale; reziduurile nu sunt perfect normale, iar erorile standard sunt subestimate, pentru că aceeași țară apare în multe perechi.

Paragraful descrie M2. Secțiunile 8-10 arată de ce M2 nu e modelul final, iar secțiunea 10 îl corectează.

## 8. Reziduurile: peste și sub model
**Reziduul** = log SCI observat − log SCI prezis de M2. Pozitiv: perechea e mai legată decât prezic distanța, migrația și granița. Negativ: mai puțin legată. 10 la puterea reziduului spune de câte ori.

In [ ]:
date["prezis"] = m2.fittedvalues                          # log SCI prezis de M2
date["reziduu"] = m2.resid                                 # observat - prezis

In [ ]:
ax = sns.scatterplot(date, x="prezis", y="log_sci")
ax.axline((4, 4), slope=1, color="black")                 # pe linie: exact cât prezice modelul
ax.set(xlabel="log SCI prezis de M2", ylabel="log SCI observat");

In [ ]:
coloane = ["tara_1", "tara_2", "sci", "la_mie", "distanta_km", "reziduu"]
date.sort_values("reziduu")[coloane].head(5)              # cele mai mult sub model

In [ ]:
date.sort_values("reziduu")[coloane].tail(5)              # cele mai mult peste model

Ghicitoarea din C1, închisă: partenerii României, față de model. Întâi, tabelul:

In [ ]:
cu_ro = date[(date["tara_1"] == "RO") | (date["tara_2"] == "RO")].copy()
cu_ro["partener"] = np.where(cu_ro["tara_1"] == "RO", cu_ro["tara_2"], cu_ro["tara_1"])   # cealaltă țară din pereche
cu_ro["de_cate_ori"] = 10 ** cu_ro["reziduu"]              # 1,5 = cu 50% peste model
cu_ro.sort_values("reziduu", ascending=False)[["partener", "sci", "la_mie", "reziduu", "de_cate_ori"]].round(2)

Apoi harta: fiecare partener, colorat după reziduul lui. Albastru: peste model; roșu: sub model; gri: fără date.

In [ ]:
reziduu_ro = cu_ro.set_index("partener")["reziduu"]           # partener -> reziduu
contur["reziduu"] = contur["tara"].map(reziduu_ro)               # reziduul fiecărui partener (România: gol)
ax = contur.plot(column="reziduu", cmap="RdBu", vmin=-0.5, vmax=0.5, legend=True, figsize=(9, 7),
                 missing_kwds={"color": "#eeeeee"})
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Partenerii României: peste (albastru) și sub (roșu) model")
ax.set_axis_off()

Sub model sunt mai ales perechile cu Turcia; peste model, perechile cu Islanda. Partenerii României: Islanda, Luxemburg, Irlanda, Ungaria și Austria sunt peste model; Italia este sub model (un SCI cu aproximativ 30% mai mic decât prezice modelul), deși are cei mai mulți migranți români. Pentru Ungaria, explicația probabilă este minoritatea maghiară din România, pe care modelul nu o include. Islanda și Luxemburgul sunt țări mici: un numitor mic, ca Svalbard în C6.

**Exercițiu.** Partenerii Turciei (`"TR"`), de la cel mai mult sub model la cel mai mult peste.

In [ ]:
# 1. Perechile în care tara_1 sau tara_2 este "TR" (două condiții, cu |), ca pentru România
# 2. Ordonate după reziduu (sort_values), cu lista coloane

## 9. O analiză de rețea mică: țările ca noduri
Perechile de țări formează o **rețea**: fiecare țară este un nod, fiecare pereche aleasă este o legătură (C6). **Gradul** unui nod este numărul legăturilor lui. Două rețele mici: a vecinilor și a legăturilor peste model.

Poziția fiecărei țări pe hartă: mediana centrelor regiunilor ei (așa, Franța nu e trasă de teritoriile de peste mări).

In [ ]:
harta["x"] = centre.x.values                               # coordonatele centrului fiecărei regiuni
harta["y"] = centre.y.values
pozitie = harta.groupby("tara")[["x", "y"]].median()      # un punct pe țară
pozitii = dict(zip(pozitie.index, pozitie.values))         # țara -> (x, y), forma cerută de networkx

### 9.1. Rețeaua vecinilor
O legătură pentru fiecare pereche de țări cu graniță comună (secțiunea 1.3).

In [ ]:
V = nx.Graph()                                             # o rețea goală
V.add_edges_from(date.loc[date["granita"] == 1, ["tara_1", "tara_2"]].values)   # o legătură pe pereche de vecini
V.number_of_nodes(), V.number_of_edges()                   # câte țări, câte granițe

In [ ]:
ax = contur.plot(color="#eeeeee", edgecolor="white", figsize=(9, 7))
nx.draw(V, pozitii, ax=ax, node_size=180, node_color="#2a9d8f", edge_color="#555555", with_labels=True, font_size=7)
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Rețeaua vecinilor")

In [ ]:
pd.Series(dict(V.degree())).sort_values(ascending=False).head()     # gradul: câți vecini are fiecare țară

22 de țări au cel puțin un vecin în tabel, legate prin 30 de granițe. Germania și Austria au cei mai mulți vecini (6), apoi Franța (5). Gradul se calculează doar între cele 24 de țări care raportează migrația: în realitate, Germania are 9 vecini.

### 9.2. Rețeaua legăturilor peste model
O legătură pentru fiecare pereche de țări mult peste model: reziduu peste 0,2, adică un SCI de cel puțin 1,6 ori mai mare decât prezic distanța, migrația și granița.

In [ ]:
peste = date[date["reziduu"] > 0.2]                        # perechile mult peste model
P = nx.from_pandas_edgelist(peste, "tara_1", "tara_2")     # o legătură pe pereche
P.number_of_nodes(), P.number_of_edges()

In [ ]:
ax = contur.plot(color="#eeeeee", edgecolor="white", figsize=(9, 7))
nx.draw(P, pozitii, ax=ax, node_size=180, node_color="#e76f51", edge_color="#555555", with_labels=True, font_size=7)
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Perechile de țări peste model")

In [ ]:
pd.Series(dict(P.degree())).sort_values(ascending=False).head()     # țările cu cele mai multe legături peste model

64 de perechi de țări sunt mult peste model, între 23 de țări. Rețeaua are două centre: Islanda (21 de legături) și Luxemburgul (19), urmate de departe de Slovenia și Letonia (9). Sunt țări mici: un SCI calculat pe puțini utilizatori iese mare cu aproape oricine. Explicația probabilă ține de măsură, nu de oameni (ca Svalbard în C6). Rețeaua arată ceva ce tabelul perechilor nu arată direct: reziduurile mari nu sunt răspândite, ci se adună în jurul câtorva țări.

**De încercat.** Același grafic cu pragul 0,3 (de 2 ori peste model) și, invers, rețeaua legăturilor **sub** model (`date["reziduu"] < -0.2`). Ce țară e în centrul ei?

## 10. Un model mai bun: efectele de țară (M3)
Rețeaua din 9.2 a arătat că reziduurile mari nu sunt răspândite la întâmplare: se adună în jurul câtorva țări (Islanda, Luxemburgul, Turcia). Asta înseamnă că lipsește din model ceva **de nivelul țării**: cât de mult se folosește Facebook acolo, cât de mică e țara (numitor mic), cât de izolată e lingvistic.

**Efectele fixe** rezolvă asta fără să măsoare acești factori: fiecare țară primește o variabilă 0/1, egală cu 1 dacă țara apare în pereche. Modelul absoarbe astfel tot ce e constant la nivelul țării, iar coeficienții se citesc **în interiorul țării**: pentru aceeași țară, partenerii mai apropiați, cu mai mulți migranți, sunt mai legați?

### 10.1. Variabilele de țară
O țară se lasă deoparte, ca referință: altfel variabilele de țară ar repeta constanta modelului.

In [ ]:
tarile = sorted(set(date["tara_1"]) | set(date["tara_2"]))      # cele 24 de țări

for t in tarile[1:]:                                            # prima țară rămâne referința
    date["fe_" + t] = ((date["tara_1"] == t) | (date["tara_2"] == t)).astype(int)

efecte = " + ".join("fe_" + t for t in tarile[1:])              # "fe_BE + fe_BG + ..."
date[["tara_1", "tara_2", "fe_RO", "fe_IT"]].head()

### 10.2. Modelul M3
Aceeași formulă ca M2, plus variabilele de țară.

In [ ]:
m3 = smf.ols("log_sci ~ log_distanta + log_la_mie + granita + " + efecte, data=date).fit()

pd.DataFrame({"M2": m2.params, "M3": m3.params}).loc[predictori].round(2)   # doar cele trei variabile

In [ ]:
m2.rsquared, m3.rsquared

R² urcă de la 0,53 la 0,92, dar asta nu înseamnă că modelul a „înțeles” mult mai mult: cea mai mare parte vine din variabilele de țară, care descriu țările, nu legăturile. Ce contează sunt coeficienții celor trei variabile: distanța rămâne −0,30, migrația crește de la 0,44 la 0,57, granița de la 0,11 la 0,15.

### 10.3. Prima verificare: VIF
Ca în secțiunea 5, dar fiecare variabilă se explică acum prin celelalte două **și** prin variabilele de țară. VIF-ul variabilelor de țară nu se interpretează: ele sunt corelate între ele prin construcție.

In [ ]:
for v in predictori:
    ceilalti = " + ".join(x for x in predictori if x != v) + " + " + efecte     # ceilalți predictori și țările
    r2 = smf.ols(f"{v} ~ {ceilalti}", data=date).fit().rsquared
    print(v, "VIF =", round(1 / (1 - r2), 2))

VIF crește (distanța ajunge la aproximativ 3,8), dar rămâne sub 5: variabilele se pot separa și cu efectele de țară.

### 10.4. A doua verificare: erorile standard grupate
Erorile standard obișnuite presupun observații independente. Perechile nu sunt: Germania apare în 23 de perechi, iar perechile unei țări seamănă între ele. **Erorile grupate** (engl. *clustered standard errors*) țin cont de asta: perechile care au o țară comună sunt tratate ca un grup. Aici grupele sunt pe două dimensiuni: țara 1 și țara 2. Este o aproximare: perechea AT-RO seamănă și cu perechile în care Austria apare a doua. Corecția completă (erorile diadice) dă aceleași concluzii pe aceste date.

In [ ]:
# fiecare țară primește un număr; o pereche aparține grupului țării 1 și grupului țării 2
numar = {t: i for i, t in enumerate(tarile)}
grupe = np.column_stack([date["tara_1"].map(numar), date["tara_2"].map(numar)])

# aceleași modele, cu erorile grupate
m2g = smf.ols("log_sci ~ log_distanta + log_la_mie + granita", data=date).fit(
    cov_type="cluster", cov_kwds={"groups": grupe})
m3g = smf.ols("log_sci ~ log_distanta + log_la_mie + granita + " + efecte, data=date).fit(
    cov_type="cluster", cov_kwds={"groups": grupe})

Statistica t (coeficientul împărțit la eroarea lui) cu erorile grupate. Peste 2 în valoare absolută: efectul se deosebește de întâmplare.

In [ ]:
pd.DataFrame({"t în M2": m2g.params / m2g.bse, "t în M3": m3g.params / m3g.bse}).loc[predictori].round(1)

Cu erorile grupate, în M2 distanța (t = −1,4) și granița (t = 1,0) nu se mai deosebesc de întâmplare; rămâne doar migrația (t = 8,2). În M3, toate trei trec pragul: distanța (−3,8), migrația (21,3), granița (2,9). Efectele de țară preiau o mare parte din asemănarea dintre perechile aceleiași țări, iar estimările devin mai precise. Concluzia lui M2 despre distanță depindea deci de o presupunere greșită (perechi independente); concluzia lui M3 nu depinde de ea.

**De încercat.** Limba oficială comună (de exemplu AT-DE, BE-FR, FI-SE), construită ca granița: o variabilă 0/1. Are sens teoretic, dar se suprapune cu granița (aproape toate perechile cu limbă comună sunt vecine). Ce se întâmplă cu coeficientul graniței? Variabilele care descriu o singură țară (PIB-ul, vârsta medie) nu pot intra în M3: efectele de țară le conțin deja.

## 11. Ce se poate concluziona
- **Asociere, nu cauză.** Modelul arată că SCI merge împreună cu migrația, la aceeași distanță și graniță. Nu arată că migrația produce legăturile: legăturile pot duce și la migrație, iar ambele pot avea o cauză comună (limba, istoria).
- **Nivelul de analiză.** Rezultatele sunt despre perechi de țări, nu despre persoane.
- **Reziduurile sunt întrebări.** Un reziduu mare arată un factor pe care modelul nu îl include: o minoritate, o limbă comună, o particularitate a măsurării (țările mici, cu numitor mic).
- **Modelul final este M3.** Cu efectele de țară și cu erorile grupate, toate trei variabilele contează: la aceeași țară, o distanță de două ori mai mare merge cu un SCI cu aproximativ 19% mai mic, o rată a migranților de două ori mai mare cu unul cu aproximativ 49% mai mare, iar granița comună cu unul cu aproximativ 40% mai mare.
- **Diagnosticul a schimbat modelul.** Reziduurile și rețeaua lor au arătat ce lipsea (efectele de țară); verificările (VIF, erorile grupate) au arătat ce rezistă.

**De încercat.** Islanda și Luxemburgul au reziduuri pozitive mari cu multe țări. Care explicație se poate verifica cu datele: o proprietate a oamenilor sau o proprietate a măsurii (SCI împărțit la un număr mic de utilizatori, ca Svalbard în C6)?

## Deciziile, de la C6 la model
| Din C6 | În model |
|---|---|
| SCI: asimetrie pozitivă | log SCI |
| Svalbard: numitor foarte mic | exclus |
| distanța și granița, legate | două variabile separate: log distanță și graniță |
| migrația contează ca rată | log rata la mie, nu numărul |
| partenerii și comunitățile urmează migrația | reziduurile arată ce rămâne după migrație |
| perechile nu sunt independente | efecte fixe pe țară și erori grupate (M3) |